In [10]:
import numpy as np

def init_params(nx, nh, ny):
    

    mean = 0.0
    std_dev = 0.3
    
    W1 = np.random.normal(loc=mean, scale=std_dev, size=(nx, nh))
    W2 = np.random.normal(loc=mean, scale=std_dev, size=(nh, ny))
    
    
    b1 = np.zeros((1, nh))
    b2 = np.zeros((1, ny))
    
    # Store parameters in a dictionary for easy passing to other functions
    params = {
        "W1": W1,
        "b1": b1,
        "W2": W2,
        "b2": b2
    }
    
    return params




## 1. Pre-activation (Linear Step)

The weighted sum of inputs:

$$
Z^{[1]} = W^{[1]}X + b^{[1]}
$$

$$
Z^{[2]} = W^{[2]}A^{[1]} + b^{[2]}
$$


## 2. Activation

Applying a non-linear activation function:

$$
A^{[1]} = \tanh(Z^{[1]})
$$

$$
A^{[2]} = \text{softmax}(Z^{[2]})
$$

---

## Mapping to the MLP Architecture

### Hidden Layer – Linear Step
$$
Z^{[1]} = W^{[1]}X + b^{[1]}
$$

The hidden layer receives input data $X$, applies weights $W^{[1]}$, and adds bias $b^{[1]}$.

### Hidden Layer – Activation
$$
A^{[1]} = \tanh(Z^{[1]})
$$

Introduces non-linearity using the Tanh activation function.

### Output Layer – Linear Step
$$
Z^{[2]} = W^{[2]}A^{[1]} + b^{[2]}
$$

Uses hidden activations $A^{[1]}$ as input.

### Output Layer – Activation
$$
A^{[2]} = \text{softmax}(Z^{[2]})
$$

Produces a probability distribution for multi-class classification.

In [11]:
import numpy as np

def softmax(Z):
    exp_Z = np.exp(Z - np.max(Z, axis=1, keepdims=True))##AXIS =1 operation horizontally , keepdims = True to maintain the original shape for broadcasting
    ## WE USED THE MAXIMUM VALUE IN Z FOR NUMERICAL STABILITY TO PREVENT LARGE EXPONENTIATION WHICH CAN LEAD TO OVERFLOW
    return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)

def forward(params, X):
  
    # Extract the parameters from the dictionary
    W1 = params["W1"]
    b1 = params["b1"]
    W2 = params["W2"]
    b2 = params["b2"]
    
    # --- Hidden Layer ---
    # 1. Linear step / Pre-activation
    Z1 = np.dot(X, W1) + b1
    # 2. Activation step (Tanh)
    A1 = np.tanh(Z1)
    
    # --- Output Layer ---
    # 3. Linear step
    Z2 = np.dot(A1, W2) + b2
    # 4. Activation step (Softmax)
    A2 = softmax(Z2)
    
    # Store intermediate values needed for backpropagation
    outputs = {
        "Z1": Z1,
        "A1": A1,
        "Z2": Z2,
        "A2": A2
    }
    
    return A2, outputs

**Arguments:**
   - Yhat -- predicted probabilities from forward propagation, shape (n_batch, n_y)
   - Y -- true labels, one-hot encoded, shape (n_batch, n_y)
    
**Returns:**
   - loss -- cross-entropy loss for the batch
   - accuracy -- ratio of correctly predicted samples
    

In [12]:


import numpy as np

def loss_accuracy(Yhat, Y):
    
    # number of samples in the batch (m)
    m = Y.shape[0]
    
    # --- 1. Compute Cross-Entropy Loss ---
    
    loss = -np.sum(np.sum(Y * np.log(Yhat + 1e-8), axis=1)) / m    
    # --- 2. Compute Accuracy ---
    
    # np.argmax finds the index (the digit 0-9) with the highest value
    predictions = np.argmax(Yhat, axis=1)
    true_labels = np.argmax(Y, axis=1)
    
    # Compare predictions to true labels (creates an array of True/False), 
    # then take the mean to get the percentage of True values.
    accuracy = np.mean(predictions == true_labels)
    
    return loss, accuracy



In [13]:

# Setup network dimensions (matching MNIST dataset)
nx = 784  # 28x28 pixels
nh = 128  # Hidden layer neurons
ny = 10   # Digits 0-9
batch_size = 5 # Let's test with 5 fake images

# Create fake input data (5 images, 784 pixels each)
X_test = np.random.randn(batch_size, nx)

# Create fake one-hot encoded labels (e.g., [0,0,1,0,0,0,0,0,0,0] for digit '2')
Y_test = np.zeros((batch_size, ny))
fake_labels = [2, 7, 1, 9, 0] # The "true" digits for our 5 fake images
for i in range(batch_size):
    Y_test[i, fake_labels[i]] = 1

# Step 1: Initialize
params = init_params(nx, nh, ny)

print("1. Parameters initialized successfully.")
print(f"W1 shape: {params['W1'].shape}, b1 shape: {params['b1'].shape}, W2 shape: {params['W2'].shape}, b2 shape: {params['b2'].shape}")
# Step 2: Forward Propagation
Yhat, outputs = forward(params, X_test)
print(f"2. Forward pass complete. Output shape: {Yhat.shape} ")

# Step 3: Loss and Accuracy
loss, accuracy = loss_accuracy(Yhat, Y_test)
print(f"3. Loss: {loss:.4f}")
print(f"4. Accuracy: {accuracy * 100}%")

1. Parameters initialized successfully.
W1 shape: (784, 128), b1 shape: (1, 128), W2 shape: (128, 10), b2 shape: (1, 10)
2. Forward pass complete. Output shape: (5, 10) 
3. Loss: 5.6753
4. Accuracy: 0.0%


    Arguments:
    x -- input data of size (n_batch, n_x)
    params -- dictionary containing our parameters (W1, b1, W2, b2)
    outputs -- dictionary containing intermediate values (Z1, A1, Z2, A2)
    Y -- true "label" vector (one-hot encoded), shape (n_batch, n_y)
    
    Returns:
    grads -- dictionary containing gradients (dW1, db1, dW2, db2)


In [14]:
import numpy as np

def backward(x, params, outputs, Y):

    m = x.shape[0] # nb of semples in  batch
    
    # Extract weights and intermediate values we need
    W2 = params["W2"]
    A1 = outputs["A1"]
    A2 = outputs["A2"]
    
    # OUTPUT LAYER (Softmax & Cross Entropy)
   
    dZ2 = A2 - Y
    
    # np.dot(A1.T, dZ2) performs the matrix multiplication
    dW2 = (1 / m) * np.dot(A1.T, dZ2)
    
    # sum along axis=0 to add up the errors for each bias across the whole batch
    db2 = (1 / m) * np.sum(dZ2, axis=0, keepdims=True)
    
    # HIDDEN LAYER (Tanh)
    dA1 = np.dot(dZ2, W2.T)
    
    # The derivative of Tanh is (1 - Tanh^2). 
    # np.power(A1, 2) squares every element in the A1 matrix.
    dZ1 = dA1 * (1 - np.power(A1, 2))
    
    dW1 = (1 / m) * np.dot(x.T, dZ1)
    db1 = (1 / m) * np.sum(dZ1, axis=0, keepdims=True)
    
    grads = {
        "dW1": dW1,
        "db1": db1,
        "dW2": dW2,
        "db2": db2
    }
    
    return grads

    """
    Applies mini-batch Stochastic Gradient Descent to update parameters.  
    Arguments:
    params -- dictionary containing the current weights and biases (W1, b1, W2, b2)
    grads -- dictionary containing the gradients calculated in the backward pass (dW1, db1, dW2, db2)
    eta -- the learning rate (a small number like 0.1 or 0.01)
    
    Returns:
    params -- dictionary containing the updated weights and biases
    """
    

In [15]:
def sgd(params, grads, eta):
  
    # Update Hidden Layer parameters
    params["W1"] = params["W1"] - (eta * grads["dW1"])
    params["b1"] = params["b1"] - (eta * grads["db1"])
    
    # Update Output Layer parameters
    params["W2"] = params["W2"] - (eta * grads["dW2"])
    params["b2"] = params["b2"] - (eta * grads["db2"])
    
    return params

In [16]:
import numpy as np
import matplotlib.pyplot as plt

def train(X_train, Y_train, nx, nh, ny, epochs=50, batch_size=128, eta=0.1):
    """
    Trains the neural network using mini-batch SGD.
    """
    # 1. Initialize the Network
    params = init_params(nx, nh, ny)
    
    # Get the total number of training examples (N)
    N = X_train.shape[0] 
    
    # Lists to stock our histories for the graphs at the end
    loss_history = []
    accuracy_history = []
    
    # 2. For I = 1 ... n_epoch
    for epoch in range(epochs):
        
        # 3. Random the data (Shuffle)
        # This creates a shuffled list of indices, which we use to scramble X and Y in the exact same way
        permutation = np.random.permutation(N)
        X_shuffled = X_train[permutation]
        Y_shuffled = Y_train[permutation]
        
        epoch_loss = 0
        epoch_acc = 0
        num_batches = N // batch_size
        
        # 4. For J = 1 ... N/n_batch
        for j in range(num_batches):
            
            # 5. Load a batch of data
            start_idx = j * batch_size
            end_idx = start_idx + batch_size
            X_batch = X_shuffled[start_idx:end_idx]
            Y_batch = Y_shuffled[start_idx:end_idx]
            
            # 6. Forward Propagation on the batch
            Yhat, outputs = forward(params, X_batch)
            
            # 7. Compute the loss on the batch
            loss, acc = loss_accuracy(Yhat, Y_batch)
            epoch_loss += loss
            epoch_acc += acc
            
            # 8. Backpropagation on the batch
            grads = backward(X_batch, params, outputs, Y_batch)
            
            # 9. Apply SGD to update parameters
            params = sgd(params, grads, eta)
            
        # Calculate the average loss and accuracy for this specific epoch
        avg_loss = epoch_loss / num_batches
        avg_acc = epoch_acc / num_batches
        
        loss_history.append(avg_loss)
        accuracy_history.append(avg_acc)
        
        # Print an update every 5 epochs so we can watch it learn!
        if epoch % 5 == 0 or epoch == epochs - 1:
            print(f"Epoch {epoch:2d}/{epochs} | Loss: {avg_loss:.4f} | Accuracy: {avg_acc * 100:.2f}%")
            
    # ==========================================
    # 10. Display Accuracy and Loss graphs
    # ==========================================
    plt.figure(figsize=(12, 5))
    
    # Plot 1: Loss Graph
    plt.subplot(1, 2, 1)
    plt.plot(loss_history, color='red', linewidth=2)
    plt.title('Training Loss History')
    plt.xlabel('Epochs')
    plt.ylabel('Cross-Entropy Loss')
    plt.grid(True)
    
    # Plot 2: Accuracy Graph
    plt.subplot(1, 2, 2)
    plt.plot(accuracy_history, color='blue', linewidth=2)
    plt.title('Training Accuracy History')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # Return the fully trained parameters so we can use them to predict real data later!
    return params, loss_history, accuracy_history